# Full AIR-Bench Integration with Risk Atlas Nexus

This notebook demonstrates the comprehensive integration of AIR-Bench with Risk Atlas Nexus. AIR-Bench (AI Risk Benchmark) is a regulation-aligned safety benchmark for responsible AI development featuring a four-tiered taxonomy with 314 risk categories derived from analyzing 8 government regulations and 16 company policies worldwide.

In [ ]:
# Install Risk Atlas Nexus if not already installed
# !pip install risk-atlas-nexus

In [ ]:
# Import required libraries
from risk_atlas_nexus.library import RiskAtlasNexus
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from IPython.display import display, HTML, Markdown
import networkx as nx
from pyvis.network import Network

In [ ]:
# Initialize the Risk Atlas Nexus library
ran = RiskAtlasNexus()

## 1. Exploring the AIR-Bench Taxonomy

AIR-Bench taxonomy has a four-tier structure. Let's explore each tier to understand the organization of risks.

In [ ]:
# Get all AIR-Bench risks
airbench_risks = ran.get_airbench_risks()
print(f"Total AIR-Bench risks: {len(airbench_risks)}")

In [ ]:
# Get the tier categories structure
tier_categories = ran.get_airbench_tier_categories()

# Display each tier level
for tier_level, categories in tier_categories.items():
    print(f"\nTier {tier_level} categories ({len(categories)}):\n{'='*50}")
    for i, category in enumerate(categories[:5], 1):  # Show first 5 categories in each tier
        print(f"{i}. {category['name']}")
    if len(categories) > 5:
        print(f"...and {len(categories) - 5} more categories...")

In [ ]:
# Create a Sankey diagram to visualize the AIR-Bench four-tier structure
def create_tier_structure_visualization(ran):
    # Get risks by tier
    tier1 = ran.get_airbench_risks_by_tier(1)
    tier2 = ran.get_airbench_risks_by_tier(2)
    tier3 = ran.get_airbench_risks_by_tier(3)
    tier4 = ran.get_airbench_risks_by_tier(4)
    
    # Create a network graph
    G = nx.DiGraph()
    
    # Add nodes for each tier
    for group in tier1:
        G.add_node(group.id, level=1, label=group.name, title=group.description if hasattr(group, 'description') else '')
    
    for group in tier2:
        G.add_node(group.id, level=2, label=group.name, title=group.description if hasattr(group, 'description') else '')
        # Add edges to parent tier1 node
        if hasattr(group, 'broadMatch') and group.broadMatch:
            for parent_id in group.broadMatch:
                G.add_edge(parent_id, group.id)
    
    for group in tier3:
        G.add_node(group.id, level=3, label=group.name, title=group.description if hasattr(group, 'description') else '')
        # Add edges to parent tier2 node
        if hasattr(group, 'broadMatch') and group.broadMatch:
            for parent_id in group.broadMatch:
                G.add_edge(parent_id, group.id)
    
    # For tier 4 (risks), we'll just add a sample
    sample_risks = tier4[:20]  # Limit to 20 risks to keep visualization manageable
    for risk in sample_risks:
        G.add_node(risk.id, level=4, label=risk.name, title=risk.description if hasattr(risk, 'description') else '')
        # Add edge to parent tier3 node
        if hasattr(risk, 'isPartOf') and risk.isPartOf:
            G.add_edge(risk.isPartOf, risk.id)
    
    # Create Pyvis network
    nt = Network(notebook=True, height="800px", width="100%", directed=True)
    
    # Define colors for each level
    colors = {1: "#4CAF50", 2: "#2196F3", 3: "#FFC107", 4: "#F44336"}
    
    # Add nodes with level-based positioning
    for node, attrs in G.nodes(data=True):
        level = attrs.get('level', 1)
        nt.add_node(node, 
                   label=attrs.get('label', node),
                   title=attrs.get('title', ''),
                   color=colors.get(level, "#CCCCCC"),
                   size=25 if level < 4 else 15,
                   level=level,
                   shape="dot")
    
    # Add edges
    for source, target in G.edges():
        nt.add_edge(source, target, arrows="to")
    
    # Set physics and layout
    nt.set_options("""
    var options = {
      "physics": {
        "hierarchicalRepulsion": {
          "centralGravity": 0,
          "springLength": 100,
          "springConstant": 0.01,
          "nodeDistance": 120
        },
        "minVelocity": 0.75,
        "solver": "hierarchicalRepulsion"
      },
      "layout": {
        "hierarchical": {
          "enabled": true,
          "direction": "LR",
          "sortMethod": "directed",
          "treeSpacing": 100,
          "nodeSpacing": 150,
          "levelSeparation": 250
        }
      }
    }
    """)
    
    return nt

# Create and display the visualization
network = create_tier_structure_visualization(ran)
network.show("airbench_structure.html")

# Display the network in an iframe
display(HTML('<iframe src="airbench_structure.html" width="100%" height="800px"></iframe>'))

## 2. Finding Risk Relationships

Let's explore how AIR-Bench risks relate to other taxonomies.

In [ ]:
# Get mapping statistics across taxonomies
mapping_stats = ran.get_airbench_taxonomy_mappings_stats()

# Create a summary table
summary_data = []
for taxonomy_id, stats in mapping_stats.items():
    relationship_counts = stats.get('by_relationship', {})
    
    summary_data.append({
        'Taxonomy': taxonomy_id,
        'Total Mappings': stats.get('total_mappings', 0),
        'AIR-Bench Risks Mapped': stats.get('risks_mapped', 0),
        'Target Risks Mapped': stats.get('target_risks_mapped', 0),
        'Exact Matches': relationship_counts.get('exactMatch', 0),
        'Close Matches': relationship_counts.get('closeMatch', 0),
        'Broad Matches': relationship_counts.get('broadMatch', 0),
        'Narrow Matches': relationship_counts.get('narrowMatch', 0),
        'Related Matches': relationship_counts.get('relatedMatch', 0)
    })

pd.DataFrame(summary_data).set_index('Taxonomy')

In [ ]:
# Visualize the mapping distribution
def plot_mapping_distribution(mapping_stats):
    # Prepare data for stacked bar chart
    data = []
    for taxonomy_id, stats in mapping_stats.items():
        relationship_counts = stats.get('by_relationship', {})
        data.append({
            'Taxonomy': taxonomy_id,
            'Exact': relationship_counts.get('exactMatch', 0),
            'Close': relationship_counts.get('closeMatch', 0),
            'Broad': relationship_counts.get('broadMatch', 0),
            'Narrow': relationship_counts.get('narrowMatch', 0),
            'Related': relationship_counts.get('relatedMatch', 0)
        })
    
    # Create DataFrame
    df = pd.DataFrame(data).set_index('Taxonomy')
    
    # Create stacked bar chart
    ax = df.plot(kind='bar', stacked=True, figsize=(12, 6), 
               color=['#4CAF50', '#2196F3', '#FFC107', '#FF9800', '#F44336'])
    
    plt.title('AIR-Bench Mappings Distribution by Taxonomy', fontsize=14)
    plt.xlabel('Taxonomy', fontsize=12)
    plt.ylabel('Number of Mappings', fontsize=12)
    plt.legend(title='Relationship Type')
    plt.xticks(rotation=45)
    
    # Add total count labels on top of each bar
    for i, taxonomy in enumerate(df.index):
        total = df.loc[taxonomy].sum()
        ax.text(i, total + 5, f'Total: {int(total)}', ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Create and display the visualization
plot_mapping_distribution(mapping_stats)

## 3. Exploring Specific AIR-Bench Risks

Let's examine some specific AIR-Bench risks and their mappings to other taxonomies.

In [ ]:
# Find a sample risk - Prompt Injection
prompt_injection_risks = [risk for risk in airbench_risks if 'prompt injection' in risk.name.lower()]
if prompt_injection_risks:
    sample_risk = prompt_injection_risks[0]
    print(f"Sample Risk: {sample_risk.name} (ID: {sample_risk.id})")
    print(f"Description: {sample_risk.description}")
    
    # Get tier information for this risk
    tier_info = ran.get_airbench_tier_for_risk(sample_risk)
    print("\nTier Structure:")
    print(f"├── Tier 1: {tier_info['top_category_name']} ({tier_info['top_category']})")
    print(f"│   └── Tier 2: {tier_info['subcategory_name']} ({tier_info['subcategory']})")
    print(f"│       └── Tier 3: {tier_info['category_name']} ({tier_info['category']})")
    print(f"│           └── Tier 4: {sample_risk.name} ({sample_risk.id})")
else:
    sample_risk = airbench_risks[0]  # Fallback to first risk
    print(f"Using fallback sample risk: {sample_risk.name} (ID: {sample_risk.id})")

In [ ]:
# Get mappings for the sample risk
if sample_risk:
    mappings = ran.get_airbench_mappings_for_risk(sample_risk.id)
    
    if mappings:
        print(f"Mappings for {sample_risk.name}:\n{'-'*40}")
        
        for taxonomy_id, taxonomy_mappings in mappings.items():
            print(f"\n{taxonomy_id} ({len(taxonomy_mappings)} mappings):")
            
            for mapping in taxonomy_mappings:
                relationship = mapping['relationship']
                relationship_symbol = {
                    'exactMatch': '≡',  # Equivalent
                    'closeMatch': '≈',  # Approximately equal
                    'broadMatch': '⊃',  # Superset
                    'narrowMatch': '⊂',  # Subset
                    'relatedMatch': '~'   # Related
                }.get(relationship, '?')
                
                print(f"  {relationship_symbol} {mapping['name']} ({mapping['id']})")
                # Print truncated description
                desc = mapping.get('description', '')
                if desc and len(desc) > 100:
                    desc = desc[:100] + '...'
                if desc:
                    print(f"    {desc}")
    else:
        print(f"No mappings found for {sample_risk.name}")

In [ ]:
# Get regulatory coverage for the sample risk
if sample_risk:
    coverage = ran.get_airbench_regulatory_coverage(sample_risk.id)
    
    if coverage:
        # Create a DataFrame for coverage scores
        coverage_data = []
        for taxonomy_id, taxonomy_coverage in coverage.items():
            coverage_data.append({
                'Taxonomy': taxonomy_coverage.get('taxonomy_name', taxonomy_id),
                'Coverage Score': taxonomy_coverage.get('coverage_score', 0),
                'Total Mappings': taxonomy_coverage.get('counts', {}).get('total', 0),
                'Exact Matches': taxonomy_coverage.get('counts', {}).get('exactMatch', 0),
                'Close Matches': taxonomy_coverage.get('counts', {}).get('closeMatch', 0),
                'Related Matches': taxonomy_coverage.get('counts', {}).get('relatedMatch', 0)
            })
        
        df = pd.DataFrame(coverage_data).sort_values('Coverage Score', ascending=False)
        
        # Create a bar chart
        plt.figure(figsize=(10, 6))
        bars = plt.bar(df['Taxonomy'], df['Coverage Score'], color='#2196F3')
        
        # Add a horizontal line at 0.7 (good coverage threshold)
        plt.axhline(y=0.7, color='green', linestyle='--', alpha=0.7, label='Good Coverage (0.7)')
        # Add a horizontal line at 0.4 (minimum coverage threshold)
        plt.axhline(y=0.4, color='red', linestyle='--', alpha=0.7, label='Minimum Coverage (0.4)')
        
        plt.title(f'Regulatory Coverage Scores for "{sample_risk.name}"', fontsize=14)
        plt.xlabel('Taxonomy', fontsize=12)
        plt.ylabel('Coverage Score (0-1)', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.ylim(0, 1.1)  # Set y-axis limits
        plt.legend()
        
        # Add value labels on top of bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.05,
                    f'{height:.2f}', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
        
        # Display a more detailed table
        display(df)
    else:
        print(f"No regulatory coverage information found for {sample_risk.name}")

## 4. Taxonomy Comparison

Let's compare AIR-Bench with other taxonomies to understand the overlap and differences.

In [ ]:
# Compare AIR-Bench with IBM Risk Atlas
comparison = ran.compare_airbench_with_taxonomy("ibm-ai-risk-atlas")

# Create a summary of the comparison
if comparison and 'error' not in comparison:
    airbench_info = comparison.get('airbench', {})
    target_info = comparison.get('target', {})
    
    # Print a summary
    print(f"Comparison between {airbench_info.get('name', 'AIR-Bench')} and {target_info.get('name')}\n{'-'*80}")
    print(f"Total AIR-Bench risks: {airbench_info.get('total_risks', 0)}")
    print(f"Total {target_info.get('name')} risks: {target_info.get('total_risks', 0)}")
    print(f"AIR-Bench risks mapped: {airbench_info.get('risks_mapped', 0)} ({airbench_info.get('coverage_percentage', 0):.1f}%)")
    print(f"{target_info.get('name')} risks mapped: {target_info.get('risks_mapped', 0)} ({target_info.get('coverage_percentage', 0):.1f}%)")
    print(f"Total mappings: {comparison.get('total_mappings', 0)}")
    
    # Create a table of relationship types
    relationships = comparison.get('relationships', {})
    relationship_df = pd.DataFrame({
        'Relationship Type': list(relationships.keys()),
        'Count': list(relationships.values())
    }).sort_values('Count', ascending=False)
    
    # Create a pie chart of relationship types
    plt.figure(figsize=(8, 8))
    plt.pie(relationship_df['Count'], labels=relationship_df['Relationship Type'], 
           autopct='%1.1f%%', startangle=90, shadow=False, 
           colors=['#4CAF50', '#2196F3', '#FFC107', '#FF9800', '#F44336', '#9C27B0'])
    plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle
    plt.title(f'Relationship Types between AIR-Bench and {target_info.get("name")}', fontsize=14)
    plt.show()
    
    # Display tier coverage
    print(f"\nTier Coverage:\n{'-'*20}")
    tier_coverage = airbench_info.get('tier_coverage', {})
    for tier, coverage in tier_coverage.items():
        print(f"Tier {tier}: {coverage:.1f}%")
    
    # Create a bar chart of tier coverage
    plt.figure(figsize=(10, 6))
    tiers = [f"Tier {tier}" for tier in tier_coverage.keys()]
    coverage_values = list(tier_coverage.values())
    
    colors = ['#4CAF50', '#2196F3', '#FFC107', '#F44336']
    bars = plt.bar(tiers, coverage_values, color=colors)
    
    plt.title(f'AIR-Bench Tier Coverage in {target_info.get("name")}', fontsize=14)
    plt.xlabel('Tier Level', fontsize=12)
    plt.ylabel('Coverage Percentage', fontsize=12)
    plt.ylim(0, 105)  # Set y-axis limits
    
    # Add value labels on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 2,
                f'{height:.1f}%', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Error in comparison: {comparison.get('error', 'Unknown error')}")

In [ ]:
# Compare with multiple taxonomies
def compare_multiple_taxonomies():
    # List of taxonomies to compare
    taxonomies = ["ibm-ai-risk-atlas", "nist-ai-rmf", "owasp-llm-2.0", "mit-ai-risk-repository"]
    
    # Collect comparison data
    comparison_data = []
    for taxonomy_id in taxonomies:
        try:
            comparison = ran.compare_airbench_with_taxonomy(taxonomy_id)
            if comparison and 'error' not in comparison:
                airbench_info = comparison.get('airbench', {})
                target_info = comparison.get('target', {})
                relationships = comparison.get('relationships', {})
                
                comparison_data.append({
                    'Taxonomy': target_info.get('name', taxonomy_id),
                    'AIR-Bench Risks Mapped (%)': airbench_info.get('coverage_percentage', 0),
                    'Target Risks Mapped (%)': target_info.get('coverage_percentage', 0),
                    'Total Mappings': comparison.get('total_mappings', 0),
                    'Exact Matches': relationships.get('exactMatch', 0),
                    'Close Matches': relationships.get('closeMatch', 0),
                    'Broad Matches': relationships.get('broadMatch', 0),
                    'Narrow Matches': relationships.get('narrowMatch', 0),
                    'Related Matches': relationships.get('relatedMatch', 0)
                })
        except Exception as e:
            print(f"Error comparing with {taxonomy_id}: {str(e)}")
    
    # Create a DataFrame
    df = pd.DataFrame(comparison_data)
    
    # Create a grouped bar chart of coverage percentages
    plt.figure(figsize=(12, 6))
    
    x = range(len(df))
    width = 0.35
    
    plt.bar(x, df['AIR-Bench Risks Mapped (%)'], width, label='AIR-Bench Risk Coverage', color='#2196F3')
    plt.bar([i + width for i in x], df['Target Risks Mapped (%)'], width, label='Target Taxonomy Risk Coverage', color='#FF9800')
    
    plt.xlabel('Taxonomy', fontsize=12)
    plt.ylabel('Coverage Percentage', fontsize=12)
    plt.title('Cross-Taxonomy Coverage Comparison with AIR-Bench', fontsize=14)
    plt.xticks([i + width/2 for i in x], df['Taxonomy'], rotation=45, ha='right')
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Return the DataFrame for further analysis
    return df

# Run the comparison
comparison_df = compare_multiple_taxonomies()
display(comparison_df)

## 5. Exploring AIR-Bench by Categories

Let's examine the different risk categories in AIR-Bench and see the distribution of risks.

In [ ]:
# Get tier 1 categories and count risks in each
tier1_categories = ran.get_airbench_risks_by_tier(1)
category_counts = []

for category in tier1_categories:
    risks = ran.get_airbench_risks_by_category(category.id)
    category_counts.append({
        'Category': category.name,
        'Risk Count': len(risks),
        'ID': category.id
    })

# Create a DataFrame and visualize
category_df = pd.DataFrame(category_counts).sort_values('Risk Count', ascending=False)

plt.figure(figsize=(12, 6))
bars = plt.bar(category_df['Category'], category_df['Risk Count'], color=['#4CAF50', '#2196F3', '#FFC107', '#F44336'])

plt.title('Number of Risks by AIR-Bench Top-Level Category', fontsize=14)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Number of Risks', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Add count labels on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 1,
            str(int(height)), ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
# Examine tier 2 distribution within a selected tier 1 category
def explore_subcategories(tier1_category_id):
    # Get all tier 2 categories under this tier 1
    tier2_categories = []
    all_tier2 = ran.get_airbench_risks_by_tier(2)
    
    for category in all_tier2:
        if hasattr(category, 'broadMatch') and category.broadMatch and tier1_category_id in category.broadMatch:
            tier2_categories.append(category)
    
    # Count risks in each tier 2 category
    subcategory_counts = []
    for category in tier2_categories:
        risks = ran.get_airbench_risks_by_category(category.id)
        subcategory_counts.append({
            'Subcategory': category.name,
            'Risk Count': len(risks),
            'ID': category.id
        })
    
    # Create a DataFrame and visualize
    subcategory_df = pd.DataFrame(subcategory_counts).sort_values('Risk Count', ascending=False)
    
    if len(subcategory_df) > 0:
        # Get the parent category name
        parent_name = "Category"
        for cat in tier1_categories:
            if cat.id == tier1_category_id:
                parent_name = cat.name
                break
        
        plt.figure(figsize=(14, 6))
        bars = plt.bar(subcategory_df['Subcategory'], subcategory_df['Risk Count'], 
                      color=plt.cm.tab20(range(len(subcategory_df))))
        
        plt.title(f'Number of Risks by Subcategory in "{parent_name}"', fontsize=14)
        plt.xlabel('Subcategory', fontsize=12)
        plt.ylabel('Number of Risks', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        
        # Add count labels on top of bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    str(int(height)), ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()
        
        return subcategory_df
    else:
        print("No subcategories found for this category.")
        return None

# Select one of the top-level categories to explore
if len(category_df) > 0:
    selected_category_id = category_df.iloc[0]['ID']  # Choose the first category
    selected_category_name = category_df.iloc[0]['Category']
    print(f"Exploring subcategories of: {selected_category_name}\n{'-'*50}")
    subcategory_df = explore_subcategories(selected_category_id)
else:
    print("No categories found.")

## 6. Regulatory Alignment Analysis

AIR-Bench is designed to be aligned with regulatory frameworks. Let's analyze how well it integrates with existing taxonomies that reflect regulatory requirements.

In [ ]:
# Create a regulatory alignment report by comparing with NIST AI RMF
def analyze_regulatory_alignment():
    # Perform comparison with NIST AI RMF
    nist_comparison = ran.compare_airbench_with_taxonomy("nist-ai-rmf")
    
    if not nist_comparison or 'error' in nist_comparison:
        print("Error comparing with NIST AI RMF. Skipping regulatory alignment analysis.")
        return None
    
    # Get tier 1 categories
    tier1_categories = ran.get_airbench_risks_by_tier(1)
    
    # Analyze alignment for each tier 1 category
    alignment_data = []
    
    for category in tier1_categories:
        risks = ran.get_airbench_risks_by_category(category.id)
        
        # Count risks with NIST mappings
        mapped_count = 0
        exact_match_count = 0
        
        for risk in risks:
            mappings = ran.get_airbench_mappings_for_risk(risk.id)
            
            if 'nist-ai-rmf' in mappings and mappings['nist-ai-rmf']:
                mapped_count += 1
                
                # Check if there's an exact match
                for mapping in mappings['nist-ai-rmf']:
                    if mapping['relationship'] == 'exactMatch':
                        exact_match_count += 1
                        break
        
        # Calculate alignment percentage
        alignment_percentage = (mapped_count / len(risks)) * 100 if risks else 0
        exact_match_percentage = (exact_match_count / len(risks)) * 100 if risks else 0
        
        alignment_data.append({
            'Category': category.name,
            'Total Risks': len(risks),
            'Mapped to NIST': mapped_count,
            'Exact Matches': exact_match_count,
            'Alignment Percentage': alignment_percentage,
            'Exact Match Percentage': exact_match_percentage
        })
    
    # Create DataFrame
    alignment_df = pd.DataFrame(alignment_data)
    
    # Create a visualizaiton
    plt.figure(figsize=(12, 6))
    
    x = range(len(alignment_df))
    width = 0.35
    
    plt.bar(x, alignment_df['Alignment Percentage'], width, label='Overall Alignment', color='#2196F3')
    plt.bar([i + width for i in x], alignment_df['Exact Match Percentage'], width, label='Exact Match Alignment', color='#4CAF50')
    
    plt.xlabel('Category', fontsize=12)
    plt.ylabel('Alignment Percentage', fontsize=12)
    plt.title('AIR-Bench Regulatory Alignment with NIST AI RMF', fontsize=14)
    plt.xticks([i + width/2 for i in x], alignment_df['Category'], rotation=45, ha='right')
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    return alignment_df

# Run the regulatory alignment analysis
alignment_df = analyze_regulatory_alignment()
if alignment_df is not None:
    display(alignment_df)

## Conclusion

This notebook has demonstrated the comprehensive integration of AIR-Bench with Risk Atlas Nexus. The four-tiered structure of AIR-Bench provides a detailed framework for analyzing AI risks, and its mappings to other taxonomies enable cross-framework risk analysis.

Key features explored:

1. Exploring the four-tier structure of AIR-Bench
2. Analyzing relationships between AIR-Bench and other taxonomies
3. Examining specific risks and their mappings
4. Comparing AIR-Bench with other taxonomies
5. Exploring risk categories and their distributions
6. Analyzing regulatory alignment

AIR-Bench's regulation-aligned taxonomy provides valuable insights for ensuring AI safety and compliance with emerging government regulations and company policies.